# Frontier Search - GPU Based

In [37]:
import os
import ast
import math
import itertools
import random as py_random
import pandas as pd
from tqdm import tqdm
from dim_backprop_gpu_only import compute_dimension

In [38]:
# Helper Functions

def calculate_parameter_count(hidden: tuple, d_0: int, d_h: int) -> int:
    """Calculates the total number of weights and biases in the network."""
    sizes = [d_0] + list(hidden) + [d_h]
    return sum(m * n for m, n in zip(sizes[:-1], sizes[1:]))

def is_less_or_equal(t1: tuple, t2: tuple) -> bool:
    """Returns True if every element in t1 is <= the corresponding element in t2."""
    if len(t1) != len(t2):
        return False
    return all(a <= b for a, b in zip(t1, t2))

def evaluate_single_architecture(hidden_tuple: tuple, h: int, d_0: int, d_h: int, exponent: int):
    """Worker function to evaluate a single architecture and format the result."""
    sizes = [d_0] + list(hidden_tuple) + [d_h]
    arch_str = str(sizes)
    
    try:
        _, _, amb, _, dim, _ = compute_dimension(sizes, exponent)
        params = calculate_parameter_count(hidden_tuple, d_0, d_h)
        is_full = (dim == amb)
        
        status = "FULL " if is_full else "SHORT"
        print(f"  [{status}] {arch_str} -> Rank: {dim}/{amb} (Params: {params})")
        
        return {
            "h": h,
            "exponent": exponent,
            "architecture": arch_str,
            "num_parameters": params,
            "dimension_computed": int(dim),
            "ambient_dimension": int(amb),
            "is_full_dimension": is_full,
            "is_minimal": False 
        }
    except Exception as e:
        print(f"  [ERROR] {arch_str} failed: {e}")
        return None


In [39]:
# Core Search & Pruning Algorithm

def parameter_boundary_search(
    h_values: list, max_width=6, min_width=1, exponent=2, 
    d_0=2, d_h=1, csv_filename="architecture_search_log.csv", user_guesses=None
):
    # Set-up a warm-up computation...if this breaks then there's no point of running the rest.
    print("Warming up dimension computation...")
    compute_dimension([d_0, 2, d_h], exponent)
    print("Warm-up complete...\n")

    # Load or initialize database -- storing in .csv for readability too
    if os.path.exists(csv_filename):
        print(f"Loading existing database from '{csv_filename}'...")
        df = pd.read_csv(csv_filename)
    else:
        print("No existing database found. Starting fresh...")
        df = pd.DataFrame(columns=[
            "h", "exponent", "architecture", "num_parameters", 
            "dimension_computed", "ambient_dimension", "is_full_dimension", "is_minimal"
        ])

    evaluated_architectures = set(df["architecture"].tolist())
    new_records = []

    for h in h_values:
        print(f"\n--- FRONTIER SEARCH: h={h}, exponent={exponent} (Max Width: {max_width} & Min Width: {min_width})  ---")
        
        num_hidden = h - 1
        degree = exponent ** num_hidden
        ambient_dim = math.comb(degree + d_0 - 1, d_0 - 1) * d_h
        print(f"  Target Ambient Dimension: {ambient_dim}")

        # Extract known boundaries
        known_minimal = set()
        known_short = set()
        
        if not df.empty:
            subset_df = df[(df["h"] == h) & (df["exponent"] == exponent)]
            
            min_df = subset_df[subset_df["is_minimal"] == True]
            known_minimal.update(tuple(ast.literal_eval(a)[1:-1]) for a in min_df["architecture"])
            
            short_df = subset_df[subset_df["is_full_dimension"] == False]
            known_short.update(tuple(ast.literal_eval(a)[1:-1]) for a in short_df["architecture"])


        # 1. Evaluate User Guesses

        if user_guesses and h in user_guesses:
            for guess in user_guesses[h]:
                arch_str = str([d_0] + list(guess) + [d_h])
                if arch_str in evaluated_architectures:
                    continue
                
                print(f"  Evaluating Guess: {guess}...")
                res = evaluate_single_architecture(guess, h, d_0, d_h, exponent)
                if not res: continue
                
                evaluated_architectures.add(res["architecture"])
                new_records.append(res)
                
                if res["is_full_dimension"]:
                    known_minimal.add(guess)
                else:
                    known_short.add(guess)

        # 2. Build and Filter Candidate Pool
        # This will filter using known architectures and the trivial bound

        total_combinations = (max_width + 1 - max(1, min_width)) ** num_hidden
        print("  Generating candidate pool...")
        all_possible = itertools.product(range(max(1, min_width), max_width + 1), repeat=num_hidden)
        print("  Obtained initial pool...")
        candidate_pool = []
        rejected_count = 0
    
        for hidden in tqdm(all_possible, total=total_combinations, desc="Evaluating Architectures"):
            arch_str = str([d_0] + list(hidden) + [d_h])
            if arch_str in evaluated_architectures:
                continue
                
            params = calculate_parameter_count(hidden, d_0, d_h)
            
            # pruning obvious noncandidates
            if (params < ambient_dim or 
                any(is_less_or_equal(m, hidden) for m in known_minimal) or 
                any(is_less_or_equal(hidden, s) for s in known_short)):
                rejected_count += 1
                continue
                
            candidate_pool.append(hidden)
    
        print(f"  Pruned {rejected_count} impossible/redundant architectures.")
        print(f"  Starting sequential evaluation on {len(candidate_pool)} viable candidates.")
        

        # Shuffle queue
        py_random.shuffle(candidate_pool)
        print(f"  Finished shuffling...")


    # 3. evaluation loop
    # not currently written so I can interrupt midway through and be okay
        while candidate_pool:

            target = candidate_pool.pop()

            if any(is_less_or_equal(m, target) for m in known_minimal):
                continue
            if any(is_less_or_equal(target, s) for s in known_short):
                continue
            
            res = evaluate_single_architecture(target, h, d_0, d_h, exponent)

            if not res: 
                continue
            
            evaluated_architectures.add(res["architecture"])
            new_records.append(res)
            
            # Simply update the known sets. The lazy pruning checks above 
            # will naturally catch redundant supersets/subsets on future iterations.

            if res["is_full_dimension"]:
                known_minimal.add(target)
            else:
                known_short.add(target)       

    # 4. Save and Post-Process Minimality
    if new_records:
        new_df = pd.DataFrame(new_records).dropna(axis=1, how='all')
        df = pd.concat([df, new_df], ignore_index=True)
        print(f"\nAdded {len(new_records)} new architectures to the database.")

    print("Re-evaluating minimal filling properties (grouped by depth h and exponent)...")


    df["is_minimal"] = False 

    # Using the the know architectures, check for minimality among all possiblities

    for (h_val, exp_val), group in df[df["is_full_dimension"] == True].groupby(["h", "exponent"]):
        
        full_archs = [ast.literal_eval(arch) for arch in group["architecture"]]
        
        for idx, row in group.iterrows():

            parsed_config = ast.literal_eval(row["architecture"])

            # It is minimal if NO other full architecture is strictly smaller than it
            
            is_min = not any(
                other != parsed_config and is_less_or_equal(other, parsed_config) 
                for other in full_archs
            )
            
            df.at[idx, "is_minimal"] = is_min

    df.to_csv(csv_filename, index=False)
    print(f"Database successfully saved to '{csv_filename}'.\n")
    return df

In [ ]:
h_values_to_test = [2,3,4,5]  

# Optional: Provide known architectures to jumpstart the pruning process.
my_guesses = {}
    
d0=3
dh=2

# Run the frontier search
df_results = parameter_boundary_search(
    h_values=h_values_to_test, 
    max_width=100,        # Maximum width 
    min_width=2,        # Minimum width
    exponent=2,         # The activation degree
    d_0=d0,              # Input dimension
    d_h=dh,              # Output dimension
    csv_filename=f"../data/raw/{d0}_{dh}_architectures.csv",
    user_guesses=my_guesses
)

# Display the minimal architectures found so far
print("\n=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===")
minimal_archs = df_results[df_results['is_minimal'] == True]

if not minimal_archs.empty:
    # Sort by parameters for easier reading
    minimal_archs = minimal_archs.sort_values(by="num_parameters")
    print(minimal_archs[["architecture", "num_parameters", "dimension_computed"]].to_string(index=False))
else:
    print("No minimal full architectures found matching the criteria.")

Warming up dimension computation...
Warm-up complete...

No existing database found. Starting fresh...

--- FRONTIER SEARCH: h=2, exponent=2 (Max Width: 100 & Min Width: 2)  ---
  Target Ambient Dimension: 12
  Generating candidate pool...
  Obtained initial pool...


Evaluating Architectures: 100%|██████████| 99/99 [00:00<00:00, 249690.98it/s]

  Pruned 1 impossible/redundant architectures.
  Starting sequential evaluation on 98 viable candidates.
  Finished shuffling...
  [FULL ] [3, 15, 2] -> Rank: 12/12 (Params: 75)
  [FULL ] [3, 10, 2] -> Rank: 12/12 (Params: 50)
  [FULL ] [3, 5, 2] -> Rank: 12/12 (Params: 25)
  [FULL ] [3, 4, 2] -> Rank: 12/12 (Params: 20)


  [FULL ] [3, 3, 2] -> Rank: 12/12 (Params: 15)

--- FRONTIER SEARCH: h=3, exponent=2 (Max Width: 100 & Min Width: 2)  ---
  Target Ambient Dimension: 30
  Generating candidate pool...
  Obtained initial pool...


Evaluating Architectures: 100%|██████████| 9801/9801 [00:00<00:00, 271304.79it/s]

...

  [FULL ] [3, 20, 69, 2] -> Rank: 30/30 (Params: 1578)
  [FULL ] [3, 14, 61, 2] -> Rank: 30/30 (Params: 1018)
  [FULL ] [3, 12, 8, 2] -> Rank: 30/30 (Params: 148)
  [SHORT] [3, 47, 3, 2] -> Rank: 21/30 (Params: 288)
  [FULL ] [3, 95, 7, 2] -> Rank: 30/30 (Params: 964)
  [SHORT] [3, 25, 4, 2] -> Rank: 28/30 (Params: 183)
  [FULL ] [3, 5, 81, 2] -> Rank: 30/30 (Params: 582)
  [FULL ] [3, 76, 7, 2] -> Rank: 30/30 (Params: 774)
  [FULL ] [3, 85, 5, 2] -> Rank: 30/30 (Params: 690)
  [SHORT] [3, 3, 60, 2] -> Rank: 18/30 (Params: 309)
  [SHORT] [3, 4, 68, 2] -> Rank: 28/30 (Params: 420)
  [FULL ] [3, 6, 39, 2] -> Rank: 30/30 (Params: 330)
  [SHORT] [3, 80, 3, 2] -> Rank: 21/30 (Params: 486)
  [SHORT] [3, 93, 2, 2] -> Rank: 14/30 (Params: 469)
  [FULL ] [3, 10, 30, 2] -> Rank: 30/30 (Params: 390)
  [FULL ] [3, 5, 47, 2] -> Rank: 30/30 (Params: 344)
  [FULL ] [3, 7, 35, 2] -> Rank: 30/30 (Params: 336)
  [SHORT] [3, 66, 4, 2] -> Rank: 28/30 (Params: 470)
  [SHORT] [3, 4, 80, 2] -> Rank: 28/30 (P

Evaluating Architectures: 100%|██████████| 970299/970299 [00:02<00:00, 375343.28it/s]


  Pruned 441 impossible/redundant architectures.
  Starting sequential evaluation on 969858 viable candidates.
  Finished shuffling...
  [FULL ] [3, 80, 54, 12, 2] -> Rank: 90/90 (Params: 5232)
  [FULL ] [3, 70, 99, 9, 2] -> Rank: 90/90 (Params: 8049)
  [FULL ] [3, 16, 10, 68, 2] -> Rank: 90/90 (Params: 1024)
  [FULL ] [3, 10, 49, 63, 2] -> Rank: 90/90 (Params: 3733)
  [FULL ] [3, 39, 80, 18, 2] -> Rank: 90/90 (Params: 4713)
  [SHORT] [3, 38, 42, 4, 2] -> Rank: 64/90 (Params: 1886)
  [FULL ] [3, 54, 67, 36, 2] -> Rank: 90/90 (Params: 6264)
  [FULL ] [3, 28, 14, 64, 2] -> Rank: 90/90 (Params: 1500)
  [SHORT] [3, 64, 31, 5, 2] -> Rank: 80/90 (Params: 2341)
  [FULL ] [3, 7, 93, 46, 2] -> Rank: 90/90 (Params: 5042)
  [FULL ] [3, 72, 45, 42, 2] -> Rank: 90/90 (Params: 5430)
  [FULL ] [3, 15, 22, 35, 2] -> Rank: 90/90 (Params: 1215)
  [FULL ] [3, 62, 61, 32, 2] -> Rank: 90/90 (Params: 5984)
  [FULL ] [3, 86, 71, 10, 2] -> Rank: 90/90 (Params: 7094)
  [SHORT] [3, 59, 2, 44, 2] -> Rank: 16/90 

Evaluating Architectures: 100%|██████████| 96059601/96059601 [04:53<00:00, 327210.30it/s]


  Pruned 85775 impossible/redundant architectures.
  Starting sequential evaluation on 95973826 viable candidates.
  Finished shuffling...
  [SHORT] [3, 55, 53, 6, 35, 2] -> Rank: 126/306 (Params: 3678)
  [FULL ] [3, 84, 46, 53, 41, 2] -> Rank: 306/306 (Params: 8809)
  [FULL ] [3, 21, 31, 56, 98, 2] -> Rank: 306/306 (Params: 8134)
  [FULL ] [3, 67, 88, 42, 12, 2] -> Rank: 306/306 (Params: 10321)
  [FULL ] [3, 94, 28, 65, 42, 2] -> Rank: 306/306 (Params: 7548)
  [FULL ] [3, 47, 23, 85, 40, 2] -> Rank: 306/306 (Params: 6657)
  [SHORT] [3, 72, 54, 11, 68, 2] -> Rank: 286/306 (Params: 5582)
  [FULL ] [3, 74, 16, 86, 63, 2] -> Rank: 306/306 (Params: 8326)
  [SHORT] [3, 2, 3, 64, 8, 2] -> Rank: 22/306 (Params: 732)
  [FULL ] [3, 58, 33, 39, 20, 2] -> Rank: 306/306 (Params: 4195)
  [FULL ] [3, 43, 97, 96, 9, 2] -> Rank: 306/306 (Params: 14494)
  [SHORT] [3, 43, 72, 6, 45, 2] -> Rank: 126/306 (Params: 4017)
  [FULL ] [3, 38, 19, 43, 82, 2] -> Rank: 306/306 (Params: 5343)
  [SHORT] [3, 2, 43, 1